# Beam-Column Design to SANS 10162-1

**Worked Example E6.1** — Beam-column subject to strong-axis bending

*Reference: Design of Structural Steelwork to SANS 10162, Edition 3 (2013)*


## Setup

Run this cell first. On Google Colab it installs the required libraries and downloads the section property database from GitHub. On a local Jupyter install with the libraries already present, it just loads them.

In [ ]:
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules

required = ['handcalcs', 'forallpeople', 'pandas']
missing = []
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing: {missing}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=True)

# Fetch section CSVs from GitHub if not present locally
REPO_RAW = "https://raw.githubusercontent.com/SAEng-design/beam-column-notebook/main/data"
for fname in ['h_sections.csv', 'i_sections.csv']:
    if not os.path.exists(fname):
        import urllib.request
        url = f"{REPO_RAW}/{fname}"
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, fname)

print("Setup complete.")


In [ ]:
import handcalcs.render
from handcalcs.decorator import handcalc
import forallpeople as si
import pandas as pd
from math import pi, sqrt
from datetime import date
from IPython.display import HTML, display

si.environment('structural', top_level=True)
print("Libraries loaded.")


## Project Information

**Edit the dictionary below for each new calculation.** Everything else in the notebook reads from these values.

In [ ]:
PROJECT = {
    "project_name":   "Industrial Warehouse Phase 2",
    "project_number": "VIKO-2026-014",
    "location":       "Pretoria, South Africa",
    "client":         "Example Client (Pty) Ltd",
    "member_type":    "Internal Braced Column",
    "member_code":    "COL-B2-12",
    "engineer":       "J. Engineer, Pr.Eng",
    "checked_by":     "S. Senior, Pr.Eng",
    "approved_by":    "M. Director, Pr.Eng",
    "revision":       "Rev A",
    "date":           date.today().isoformat(),
    "calc_ref":       "CALC-E6.1-001",
}


In [ ]:
LOGO_B64 = "..."

header_html = f"""
<div style="border:2px solid #4A7BB7; border-radius:6px; padding:14px 18px;
            margin:6px 0 18px 0; font-family: Arial, Helvetica, sans-serif;
            background:linear-gradient(to right,#F4F8FC,#FFFFFF);">
  <table style="width:100%; border-collapse:collapse;">
    <tr>
      <td style="width:280px; vertical-align:middle; padding-right:18px;">
        <img src="data:image/png;base64,{LOGO_B64}"
             style="width:260px; max-width:100%; display:block;" alt="ViKO Consulting Engineers"/>
      </td>
      <td style="vertical-align:middle; border-left:1px solid #B5C9E2; padding-left:18px;">
        <div style="font-size:18px; font-weight:700; color:#2C4A78; margin-bottom:4px;">
          {PROJECT['project_name']}
        </div>
        <div style="font-size:13px; color:#444; margin-bottom:8px;">
          {PROJECT['client']} &nbsp;|&nbsp; {PROJECT['location']}
        </div>
        <table style="font-size:12px; color:#333; border-collapse:collapse;">
          <tr>
            <td style="padding:1px 14px 1px 0;"><b>Project No.:</b> {PROJECT['project_number']}</td>
            <td style="padding:1px 14px 1px 0;"><b>Calc Ref.:</b> {PROJECT['calc_ref']}</td>
            <td style="padding:1px 0;"><b>Revision:</b> {PROJECT['revision']}</td>
          </tr>
          <tr>
            <td style="padding:1px 14px 1px 0;"><b>Member:</b> {PROJECT['member_code']}</td>
            <td style="padding:1px 14px 1px 0;"><b>Type:</b> {PROJECT['member_type']}</td>
            <td style="padding:1px 0;"><b>Date:</b> {PROJECT['date']}</td>
          </tr>
        </table>
      </td>
    </tr>
  </table>
  <table style="width:100%; margin-top:10px; border-top:1px solid #B5C9E2;
                padding-top:8px; font-size:12px; color:#333; border-collapse:collapse;">
    <tr>
      <td style="padding:3px 0;"><b>Engineer:</b> {PROJECT['engineer']}</td>
      <td style="padding:3px 0;"><b>Checked by:</b> {PROJECT['checked_by']}</td>
      <td style="padding:3px 0;"><b>Approved by:</b> {PROJECT['approved_by']}</td>
    </tr>
  </table>
</div>
"""
display(HTML(header_html))


## Problem Statement

A segment of a braced column between two floors in a multi-storey building is subject to the following ultimate design loads:

| Load | Value |
|---|---|
| Axial compressive load, $C_u$ | $2\,200~\text{kN}$ |
| Bending moment at top (major axis), $M_{u,\text{top}}$ | $180~\text{kNm}$ |
| Bending moment at bottom (major axis), $M_{u,\text{bot}}$ | $240~\text{kNm}$ |
| Column unbraced length, $L$ | $3.2~\text{m}$ |
| Trial section | $305 \times 305 \times 118$ kg/m H-section |
| Steel grade | S355JR ($f_y = 350~\text{MPa}$) |

**Three checks required (Clause 13.8.2):**
1. Cross-sectional strength
2. Overall member strength  
3. Lateral-torsional buckling


## Section Property Database

The H-section database (`h_sections.csv`) follows the SAISC Red Book convention. Units: `A` in ×10³ mm², `Ix`/`Iy` in ×10⁶ mm⁴, `Sx`/`Sy` (plastic moduli) in ×10³ mm³, `J` in ×10³ mm⁴, `Cw` in ×10⁹ mm⁶, linear dimensions in mm.

In [ ]:
sections = pd.read_csv('h_sections.csv', sep=';', decimal=',').set_index('designation')
print(f"Loaded {len(sections)} H-sections")
sections.head()


In [ ]:
# === Trial section selection ===
SECTION = '305x305x118'
sec = sections.loc[SECTION]
print(f"Selected: {SECTION}")
sec


## Design Inputs

In [ ]:
%%render
# Design loads
C_u = 2200 * si.kN
M_u_top = 180 * si.kN * si.m
M_u_bot = 240 * si.kN * si.m

# Column geometry
L = 3.2 * si.m

# Material (Grade S355JR, fy = 350 MPa per worked example)
f_y = 350 * si.MPa
E   = 200000 * si.MPa
G   = 77000 * si.MPa

# Resistance factor
phi = 0.9


In [ ]:
%%render
# Section properties — loaded from h_sections.csv, converted to SI
A     = float(sec['A'])  * 1e3 * si.mm**2
I_x   = float(sec['Ix']) * 1e6 * si.mm**4
I_y   = float(sec['Iy']) * 1e6 * si.mm**4
J     = float(sec['J'])  * 1e3 * si.mm**4
C_w   = float(sec['Cw']) * 1e9 * si.mm**6
Z_plx = float(sec['Sx']) * 1e3 * si.mm**3
b_f   = float(sec['b'])  * si.mm
t_f   = float(sec['tf']) * si.mm
t_w   = float(sec['tw']) * si.mm
h_w   = float(sec['d'])  * si.mm
r_x   = float(sec['rx']) * si.mm
r_y   = float(sec['ry']) * si.mm


## Cross-Section Classification (Clause 11)

### Compression flange outstand (Clause 11.3.1(c), Table 4)

In [ ]:
%%render
fy_MPa = 350   # numeric value of f_y in MPa, for limit formulas

flange_ratio    = (b_f / 2) / t_f
lambda_Class1_f = 145 / sqrt(fy_MPa)
lambda_Class2_f = 170 / sqrt(fy_MPa)


In [ ]:
import math
fr  = (float(sec['b'])/2)/float(sec['tf'])
c1f = 145/math.sqrt(350)
c2f = 170/math.sqrt(350)
print(f"  Flange b/(2 t_f) = {fr:.2f}")
print(f"  Class 1 limit    = {c1f:.2f}  →  {'within' if fr<=c1f else 'exceeded'}")
print(f"  Class 2 limit    = {c2f:.2f}  →  {'within' if fr<=c2f else 'exceeded'}")
print(f"\n  → Compression flange is Class 2")


### Web in combined axial compression and bending (Clause 11.3.2(c), Table 4)

In [ ]:
%%render
web_ratio       = h_w / t_w
C_y             = A * f_y
lambda_Class1_w = (1100 / sqrt(fy_MPa)) * (1 - 0.39 * C_u / (phi * C_y))


In [ ]:
wr  = float(sec['d'])/float(sec['tw'])
Cy_num = float(sec['A'])*1e3 * 350
limw = (1100/math.sqrt(350)) * (1 - 0.39 * 2200e3/(0.9*Cy_num))
print(f"  Web h/t_w     = {wr:.2f}")
print(f"  Class 1 limit = {limw:.2f}  →  {'within' if wr<=limw else 'exceeded'}")
print(f"\n  → Web is Class 1")
print(f"  → Overall section classification: CLASS 2")


## Check (i) — Cross-Sectional Strength

**Clause 13.8.2:**
$$\frac{C_u}{C_{rx}} + \frac{0.85\,U_{1x}\,M_{ux}}{M_{rx}} \le 1.0$$

### Compressive resistance $C_{rx}$ — Clause 13.3.1 with $\lambda = 0$

In [ ]:
%%render
# For cross-section check, lambda = 0 → C_rx reduces to phi * A * f_y
C_rx = phi * A * f_y


### Moment resistance $M_{rx}$ — Clause 13.5(a), Class 2 section

In [ ]:
%%render
M_rx = phi * Z_plx * f_y


### Moment amplification factor $U_{1x}$ — Clause 13.8.4

For a braced frame, $\beta = 0.60$. The moment-gradient factor from Clause 13.8.5(a):
$$\omega_{1x} = 0.6 - 0.4\kappa \ge 0.4 \quad ; \quad \kappa = +\frac{M_{u,\text{smaller}}}{M_{u,\text{larger}}} \text{ for double curvature}$$

In [ ]:
%%render
beta = 0.60
kappa = + (M_u_top / M_u_bot)
omega_1x_calc = 0.6 - 0.4 * kappa
omega_1x = max(omega_1x_calc, 0.4)


In [ ]:
%%render
# Euler buckling load about strong axis
KL_x = 1.0 * L
C_ex = (pi**2 * E * I_x) / (KL_x**2)

# Amplification factor (Clause 13.8.4)
U_1x_calc = omega_1x / (1 - C_u / C_ex)

# For cross-section check, U_1x shall not be < 1.0
U_1x = max(U_1x_calc, 1.0)


### Interaction equation

In [ ]:
%%render
M_ux = M_u_bot   # larger end moment governs

ratio_axial_i   = C_u / C_rx
ratio_bending_i = (0.85 * U_1x * M_ux) / M_rx
interaction_i   = ratio_axial_i + ratio_bending_i


In [ ]:
verdict = "OK" if 0.80 <= 1.0 else "FAIL"
print(f"  Interaction (i)  =  0.465 + 0.332  =  0.80   <= 1.0   ->   {verdict}")
print(f"  -> Cross-sectional strength is adequate")


## Check (ii) — Overall Member Strength

**Clause 13.8.2** — for uniaxial strong-axis bending in a braced frame, the compressive resistance is based on weak-axis flexural buckling ($C_r = C_{ry}$).

### Weak-axis compressive resistance $C_{ry}$ — Clause 13.3.1

In [ ]:
%%render
K_y = 1.0
slenderness_y = (K_y * L) / r_y
f_ey = (pi**2 * E) / (slenderness_y**2)
lambda_y = sqrt(f_y / f_ey)

# n = 1.34 for hot-rolled H-sections
n_param = 1.34
C_ry = phi * A * f_y * (1 + lambda_y**(2*n_param))**(-1/n_param)


### Interaction equation

For overall member check, the calculated $U_{1x}$ is used (not bounded below by 1.0):

In [ ]:
%%render
U_1x_overall = U_1x_calc   # = 0.417, used as-is

ratio_axial_ii   = C_u / C_ry
ratio_bending_ii = (0.85 * U_1x_overall * M_ux) / M_rx
interaction_ii   = ratio_axial_ii + ratio_bending_ii


In [ ]:
verdict = "OK" if 0.63 <= 1.0 else "FAIL"
print(f"  Interaction (ii)  =  0.487 + 0.138  =  0.63   <= 1.0   ->   {verdict}")
print(f"  -> Overall member strength is adequate")


## Check (iii) — Lateral-Torsional Buckling

### Compressive resistance for LTB check — Clause 13.3.2

Take the lesser of weak-axis flexural buckling and torsional/torsional-flexural buckling stress.

In [ ]:
%%render
# Flexural buckling stress about weak axis (same as in (ii) but renamed)
f_ey_LTB = (pi**2 * E) / ((K_y * L / r_y)**2)

# Polar radius of gyration about the shear centre.
# For doubly-symmetric H-section: x_0 = y_0 = 0  →  r_0^2 = r_x^2 + r_y^2
r_0_sq = r_x**2 + r_y**2

# Torsional buckling stress (K_z = 1.0 conservative)
K_z = 1.0
KL_z = K_z * L
f_ez = ((pi**2 * E * C_w) / (KL_z**2) + G * J) / (A * r_0_sq)

# Critical elastic stress
f_e_LTB = min(f_ey_LTB, f_ez)


In [ ]:
%%render
lambda_LTB = sqrt(f_y / f_e_LTB)
C_r_LTB = phi * A * f_y * (1 + lambda_LTB**(2*n_param))**(-1/n_param)


### LTB moment resistance $M_{rx,\text{LTB}}$ — Clause 13.6(a)

For double curvature: $\omega_2 = 1.75 + 1.05\kappa + 0.3\kappa^2 \le 2.5$

In [ ]:
%%render
omega_2_calc = 1.75 + 1.05 * kappa + 0.3 * kappa**2
omega_2 = min(omega_2_calc, 2.5)


In [ ]:
%%render
# Elastic critical moment
KL_LTB = K_y * L
M_cr = (omega_2 * pi / KL_LTB) * sqrt(
    E * I_y * G * J + (pi * E / L)**2 * I_y * C_w
)


In [ ]:
%%render
# Plastic moment (unfactored)
M_px = Z_plx * f_y

# Threshold: 0.67 * M_px < M_cr → inelastic LTB formula governs
threshold_LTB = 0.67 * M_px


In [ ]:
%%render
# Inelastic LTB — Clause 13.6(a)(i)
M_rx_LTB_calc = 1.15 * phi * M_px * (1 - 0.28 * M_px / M_cr)
M_rx_LTB = min(M_rx_LTB_calc, phi * M_px)


### Interaction equation for LTB

In [ ]:
%%render
ratio_axial_iii   = C_u / C_r_LTB
ratio_bending_iii = (0.85 * U_1x_overall * M_ux) / M_rx_LTB
interaction_iii   = ratio_axial_iii + ratio_bending_iii


In [ ]:
verdict = "OK" if 0.87 <= 1.0 else "FAIL"
print(f"  Interaction (iii)  =  0.534 + 0.332  =  0.87   <= 1.0   ->   {verdict}")
print(f"  -> Lateral-torsional buckling resistance is adequate")


## Summary

In [ ]:
summary_html = f"""
<div style="border:2px solid #4A7BB7; border-radius:6px; padding:14px 18px;
            margin-top:18px; font-family: Arial, sans-serif;
            background:#F4F8FC;">
  <div style="font-size:16px; font-weight:700; color:#2C4A78; margin-bottom:8px;">
    Design Summary — Member {PROJECT['member_code']}
  </div>
  <table style="width:100%; border-collapse:collapse; font-size:13px;">
    <tr style="background:#D6E3F3;">
      <th style="text-align:left; padding:6px 10px; border:1px solid #B5C9E2;">Check</th>
      <th style="text-align:left; padding:6px 10px; border:1px solid #B5C9E2;">Clause</th>
      <th style="text-align:right; padding:6px 10px; border:1px solid #B5C9E2;">Utilisation</th>
      <th style="text-align:center; padding:6px 10px; border:1px solid #B5C9E2;">Status</th>
    </tr>
    <tr>
      <td style="padding:5px 10px; border:1px solid #B5C9E2;">(i) Cross-sectional strength</td>
      <td style="padding:5px 10px; border:1px solid #B5C9E2;">13.8.2</td>
      <td style="text-align:right; padding:5px 10px; border:1px solid #B5C9E2;">0.80</td>
      <td style="text-align:center; padding:5px 10px; border:1px solid #B5C9E2; color:#2E7D32;"><b>OK</b></td>
    </tr>
    <tr>
      <td style="padding:5px 10px; border:1px solid #B5C9E2;">(ii) Overall member strength</td>
      <td style="padding:5px 10px; border:1px solid #B5C9E2;">13.8.2</td>
      <td style="text-align:right; padding:5px 10px; border:1px solid #B5C9E2;">0.63</td>
      <td style="text-align:center; padding:5px 10px; border:1px solid #B5C9E2; color:#2E7D32;"><b>OK</b></td>
    </tr>
    <tr>
      <td style="padding:5px 10px; border:1px solid #B5C9E2;">(iii) Lateral-torsional buckling</td>
      <td style="padding:5px 10px; border:1px solid #B5C9E2;">13.8.2 / 13.6(a)</td>
      <td style="text-align:right; padding:5px 10px; border:1px solid #B5C9E2;">0.87</td>
      <td style="text-align:center; padding:5px 10px; border:1px solid #B5C9E2; color:#2E7D32;"><b>OK</b></td>
    </tr>
  </table>
  <div style="margin-top:12px; padding:10px 14px; background:#E8F5E9;
              border-left:4px solid #2E7D32; font-size:14px; font-weight:600; color:#1B5E20;">
    Section {SECTION} ({sec['m']:.0f} kg/m), Grade S355JR — ADEQUATE
  </div>
  <div style="margin-top:10px; font-size:11px; color:#666;">
    Calculation prepared per SANS 10162-1. Engineer: {PROJECT['engineer']}.
    Checked by: {PROJECT['checked_by']}. Approved by: {PROJECT['approved_by']}.
    Calc Ref. {PROJECT['calc_ref']} ({PROJECT['revision']}), {PROJECT['date']}.
  </div>
</div>
"""
display(HTML(summary_html))


## Export to PDF

Run this cell to generate a PDF copy of the completed calculation for archiving. The PDF is saved alongside the notebook and (on Colab) downloads automatically.

> The first PDF export in a fresh Colab session takes ~30 seconds while the headless Chromium engine is set up.

In [ ]:
import subprocess, sys

# Notebook filename — auto-detect on Colab, fallback for local Jupyter
try:
    from google.colab import _message
    info = _message.blocking_request('get_ipynb', timeout_sec=5)
    nb_name = info['ipynb']['metadata']['colab']['name']
    nb_path = nb_name if nb_name.endswith('.ipynb') else nb_name + '.ipynb'
except Exception:
    nb_path = 'E6_1_beam_column_SANS10162.ipynb'   # rename if your saved copy differs

# Build output filename from project metadata
out_name = f"{PROJECT['calc_ref']}_{PROJECT['member_code']}_{PROJECT['revision'].replace(' ','')}"
out_name = out_name.replace('/', '-').replace(' ', '_')

print(f"Exporting → {out_name}.pdf")
subprocess.run([
    sys.executable, '-m', 'nbconvert',
    '--to', 'webpdf',
    '--allow-chromium-download',
    '--output', out_name,
    nb_path
], check=True)

print(f"Done. File saved as: {out_name}.pdf")

# Trigger browser download on Colab
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(f"{out_name}.pdf")


---

## Appendix — Notes for the Design Team

### Workflow

1. Open this notebook from the GitHub repo via the **Open in Colab** badge.
2. In Colab: **File → Save a copy in Drive**, then rename per job (e.g. `COL-B2-12_E6_1.ipynb`).
3. Edit the `PROJECT` dictionary in Section 2 with project particulars.
4. Adjust design loads in **Design Inputs** and select your trial section.
5. **Runtime → Run all**.
6. Review results, then run the **Export to PDF** cell to generate the archive copy.

### Changing the trial section

Edit the `SECTION` variable. To list available designations:

```python
print(list(sections.index))
```

### Caveats / scope

- This notebook implements **Example E6.1** specifically (uniaxial strong-axis bending, braced frame, double curvature). Other load cases — biaxial bending, sway frames, single curvature — require changes to $\omega_{1x}$, $\omega_2$, and the U-factor logic (Clauses 13.8.4–13.8.5).
- Material is set to $f_y = 350~\text{MPa}$ (Grade S355JR per the worked example). For other grades, edit `f_y` in **Design Inputs** and check thickness vs. yield reductions in the relevant material standard.
- Buckling parameter $n = 1.34$ assumes a hot-rolled H-section (Clause 13.3.1). For welded sections, $n = 2.24$.
- $K_z = 1.0$ is conservative for LTB — a lower value may be justified with restraint details (Clause 13.6).

### References

- SANS 10162-1:2011 — *The structural use of steel — Part 1: Limit-states design of hot-rolled steelwork*
- *Design of Structural Steelwork to SANS 10162*, Edition 3 (2013), SAISC — Example E6.1
- SAISC Red Book — Section property tables
- `handcalcs` — [github.com/connorferster/handcalcs](https://github.com/connorferster/handcalcs)
- `forallpeople` — [github.com/connorferster/forallpeople](https://github.com/connorferster/forallpeople)
